In [1]:
import os
import openai

OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")

# 클라이언트 초기화
client = openai.OpenAI(
    base_url=OLLAMA_API_URL, # OpenAI API 대신 ollama 사용
    api_key="ollama-key" # 필수로 문자열을 요구하므로 주석 해제 유지
)

# Ollama 서버의 모델 목록 조회
models = client.models.list()

# 사용 가능한 모델의 ID(이름) 출력
print("=== 사용 가능한 모델 목록 ===")
for model in models.data:
    print("-", model.id)


=== 사용 가능한 모델 목록 ===
- gemma4:e4b
- gemma4:e2b
- qwen3.5:4b
- qwen3.5:9b
- nemotron-3-nano:4b
- functiongemma:latest
- translategemma:12b
- translategemma:4b
- nomic-embed-text-v2-moe:latest
- olmo-3:7b
- ministral-3:14b
- ministral-3:8b
- ministral-3:3b
- qwen3-vl:8b
- qwen3-vl:4b
- qwen3-vl:2b
- qwen3-embedding:0.6b
- embeddinggemma:300m
- gemma3n:e2b
- gemma3n:e4b
- linux6200/bge-reranker-v2-m3:latest
- gemma3:4b
- gemma3:1b
- gemma3:12b
- bona/bge-m3-korean:latest
- shieldgemma:latest
- llama-guard3:8b
- llama3.2-vision:11b
- nomic-embed-text:latest
- codegemma:latest


In [2]:
import openai
import json
import requests

# 1. 영화 API 호출 함수
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    data = response.json()
    
    print("\n" + "="*50)
    print("🍿 [현재 인기 영화 요약 목록]")
    
    # API에 따라 data가 바로 리스트일 수도 있고, data['results'] 안에 있을 수도 있습니다.
    # 보여주신 예시는 바로 리스트 형태인 것으로 보여 리스트로 가정합니다.
    movies = data if isinstance(data, list) else data.get('results', [])
    summary_result = ""
    
    # 데이터가 너무 많을 수 있으니 상위 5개만 예시로 출력해 봅니다.
    for idx, movie in enumerate(movies[:5], start=1):
        # 'title'이 없으면 'original_title' 사용
        movie_id = movie.get('id', movie.get('movie_id', 'id 없음'))
        title = movie.get('title', movie.get('original_title', '제목 없음'))
        release_date = movie.get('release_date', '개봉일 미상')
        rating = movie.get('vote_average', '평점 없음')
        
        summary_result += f" {idx}. [ID: {movie_id}] 🎬 {title} (개봉: {release_date} | ⭐ {rating})"
    print(summary_result)
    print("="*50 + "\n")
    
    return summary_result

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

def get_movie_similar(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return response.json()

# ⭐️ 추가: 함수 이름(문자열)과 실제 파이썬 함수를 매핑해주는 딕셔너리
available_functions = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_movie_similar": get_movie_similar,
}

# 2. 클라이언트 초기화 (위 셀에서 했음)
# client = openai.OpenAI(
#     base_url=OLLAMA_API_URL,
#     api_key="ollama-key" 
# )

# 3. 도구(Tools) 정의 (이전과 동일)
tools = [
     {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 ID를 가진 영화의 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 고유 ID"}}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 ID를 가진 영화의 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 고유 ID"}}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "특정 ID를 가진 영화의 유사한 영화를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 고유 ID"}}
            }
        }
    }
]

In [3]:
# 1. LLM API 호출 함수 분리 (Issue 3 해결)
def get_llm_response(messages_history):
    """현재까지의 대화 기록(messages_history)을 바탕으로 LLM의 응답을 가져옵니다."""
    response = client.chat.completions.create(
        model="gemma4:e2b", 
        messages=messages_history,
        tools=tools,
        tool_choice="auto"
    )
    return response.choices[0].message

In [4]:
# 2. 메모리(Context) 초기화 (Issue 2 해결)
# 반복문 밖에서 시스템 프롬프트를 포함하여 초기화하고, 대화 내역을 계속 누적합니다.
messages = [
    {"role": "system", "content": "당신은 사용자의 질문에 알맞은 영화 API 함수를 호출하는 Movie Expert Agent입니다."}
]

test_inputs = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "두번째 영화에 대해 더 알려줘", # 앞선 결과를 기억해야 함
    "비슷한 영화 추천해 줄래?" # 이전 영화 정보 요청 컨텍스트를 기억해야 함
]

In [5]:
print("🎬 Movie Expert Agent + 🛠️ 실제 함수 실행 테스트 시작\n" + "="*50)

for user_input in test_inputs:
    print(f"\n👤 [사용자 입력]: \"{user_input}\"")
    
    # 사용자의 입력을 메모리에 추가
    messages.append({"role": "user", "content": user_input})
    
    # 1차 LLM 호출 (함수 호출 여부 판단)
    message = get_llm_response(messages)
    
    if message.tool_calls:
        # ⭐️ 중요: LLM이 함수를 호출하겠다고 한 응답 자체도 메모리에 반드시 추가해야 합니다.
        messages.append(message)
        
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = tool_call.function.arguments
            
            print(f"🤖 [모델의 결정] '{function_name}' 함수 호출. 인자: {function_args}")
            
            function_to_call = available_functions.get(function_name)
            
            if function_to_call:
                print("   🚀 API에 실제 요청을 보냅니다...")
                try:
                    args_dict = json.loads(function_args) if function_args else {}
                    function_result = function_to_call(**args_dict)
                    print(f"   ✅ [실행 결과 (일부 발췌)]: {str(function_result)[:200]}...")
                    
                    # 3. 함수 실행 결과를 LLM에게 다시 전달
                    # role을 "tool"로 설정하고, tool_call_id를 반드시 매칭시켜야 합니다.
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": json.dumps(function_result, ensure_ascii=False) # JSON 결과를 문자열로 변환하여 전달
                    })
                    
                except Exception as e:
                    print(f"   ❌ [오류 발생]: {e}")
                    # 오류가 났을 때도 LLM에게 오류 사실을 알려주는 것이 좋습니다.
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": f"Error occurred: {str(e)}"
                    })
            else:
                print(f"   ❌ 알려지지 않은 함수가 호출되었습니다: {function_name}")
        
        # ⭐️ 도구 실행 결과를 바탕으로 최종 답변 생성 (2차 LLM 호출)
        final_message = get_llm_response(messages)
        print(f"🤖 [최종 답변]: {final_message.content}")
        
        # 최종 답변도 다음 대화를 위해 메모리에 추가
        messages.append(final_message)
        
    else:
        # 함수 호출이 필요 없는 일반 대화인 경우
        print(f"🤖 [답변]: {message.content}")
        # 어시스턴트의 답변을 메모리에 추가
        messages.append(message)

print("\n" + "="*50 + "\n🎬 테스트 종료")

🎬 Movie Expert Agent + 🛠️ 실제 함수 실행 테스트 시작

👤 [사용자 입력]: "지금 인기 있는 영화가 무엇인지 알려줘"
🤖 [모델의 결정] 'get_popular_movies' 함수 호출. 인자: {}
   🚀 API에 실제 요청을 보냅니다...

🍿 [현재 인기 영화 요약 목록]
 1. [ID: 1523145] 🎬 Your Heart Will Be Broken (개봉: 2026-03-26 | ⭐ 6.75) 2. [ID: 83533] 🎬 Avatar: Fire and Ash (개봉: 2025-12-17 | ⭐ 7.371) 3. [ID: 1327819] 🎬 Hoppers (개봉: 2026-03-04 | ⭐ 7.595) 4. [ID: 1290821] 🎬 Shelter (개봉: 2026-01-28 | ⭐ 6.789) 5. [ID: 1171145] 🎬 Crime 101 (개봉: 2026-02-11 | ⭐ 7)

   ✅ [실행 결과 (일부 발췌)]:  1. [ID: 1523145] 🎬 Your Heart Will Be Broken (개봉: 2026-03-26 | ⭐ 6.75) 2. [ID: 83533] 🎬 Avatar: Fire and Ash (개봉: 2025-12-17 | ⭐ 7.371) 3. [ID: 1327819] 🎬 Hoppers (개봉: 2026-03-04 | ⭐ 7.595) 4. [ID: 1...
🤖 [최종 답변]: 현재 인기 있는 영화 목록은 다음과 같습니다:

1.  **Your Heart Will Be Broken** (개봉: 2026-03-26 | 평점: 6.75)
2.  **Avatar: Fire and Ash** (개봉: 2025-12-17 | 평점: 7.371)
3.  **Hoppers** (개봉: 2026-03-04 | 평점: 7.595)
4.  **Shelter** (개봉: 2026-01-28 | 평점: 6.789)
5.  **Crime 101** (개봉: 2026-02-11 | 평점: 7.0)

👤 [사용자 입력]: 